# 节点 6：任务执行与数据闭环

这一节点把“建议做什么”连接到“确实做完并留下记录”：无论从状态卡片勾选，还是确认 Agent 动作，最终都走同一套安全写入流程。

## 1. 本节点目标

让一次清洗或晾晒同时形成活动历史并更新物品最近日期；重复提交不产生重复数据，写入中途失败时不留下半条记录。

## 2. 完成结果与验收

- 状态卡片可直接勾选今天已清洗或已晾晒。
- 状态页使用分类渐变卡片，并可搜索名称、分类和备注。
- 演示物品显示本地莫兰迪油画图，用户上传图片仍具有最高优先级。
- Agent 写入必须先生成待确认动作。
- 每件物品可展开最近 5 条历史，并看到记录来源。
- 同日同动作重复提交返回明确提示，不重复写入。
- 活动历史和最近日期使用同一事务。
- 模拟第二步失败后，两处数据都保持原样。
- 未来完成日期会被拒绝。
- 全量 61 项测试通过。

## 3. 本节点文件结构

```text
src/smart_laundry/repositories.py  事务写入、幂等和历史读取
src/smart_laundry/tools.py         确认写入、未来日期与重复反馈
src/smart_laundry/item_views.py    搜索、筛选和排序
app.py                            渐变卡片、手动勾选、历史展示和提示
assets/demo/                      本地莫兰迪油画演示图
tests/test_activity_records.py     历史顺序、回滚和限制测试
tests/test_tools.py                确认、重复和未来日期测试
tests/test_agent.py                Agent 只生成待确认动作测试
notebooks/07_task_execution.ipynb
```

## 4. 关键代码解释

`record_activity()` 先执行 `INSERT OR IGNORE`。表上的唯一约束是 `(item_id, action_type, performed_at)`，所以同一物品、同一种动作、同一天只能有一条记录。插入成功后，再更新 `items.last_washed_at` 或 `items.last_dried_at`。

这两条 SQL 位于同一个 `database_connection()` 上下文中：正常结束时统一提交，任意异常时统一回滚。`list_activity_records(limit=5)` 用参数化 `LIMIT` 返回最新历史。

In [ ]:
def transaction_result(first_step, second_step):
    if first_step and second_step:
        return '全部提交'
    return '全部回滚，不保留半条记录'

print(transaction_result(True, True))
print(transaction_result(True, False))

## 5. 数据流

```mermaid
flowchart LR
 A[手动勾选] --> C[record_activity]
 B[Agent 待确认] --> D[用户确认]
 D --> C
 C --> E{同日记录已存在?}
 E -->|是| F[不重复写入并提示]
 E -->|否| G[插入活动历史]
 G --> H[更新最近日期]
 H -->|两步成功| I[事务提交]
 H -->|任一步失败| J[事务回滚]
 I --> K[页面状态和历史刷新]
```

## 6. 关键概念

- **事务**：一组数据库操作要么全部成功，要么全部取消。
- **回滚**：发生错误时撤销尚未提交的修改。
- **幂等**：同一个操作重复提交，最终数据仍与提交一次相同。
- **唯一约束**：由数据库阻止重复活动记录。
- **活动历史**：保存每次操作，而不只是最后一次日期。
- **source**：区分手动勾选、Agent 确认或演示数据。

## 7. 为什么这样设计

物品表保存最近日期，读取状态速度直接；活动表保存完整历史，便于追溯。两处数据看似重复，但用事务保持一致。首版不加入复杂的撤销和历史编辑，避免破坏最近日期与历史之间的关系。页面的演示油画图只作为同名示例物品的后备图，不写入用户记录；只要用户上传照片，就优先显示真实照片，因此视觉演示不会改变数据事实。

## 8. 常见错误与排查

1. **勾选后没有变化**：查看页面错误提示，确认数据库文件可写。
2. **提示没有重复写入**：说明当天同动作已经存在，这是正常保护。
3. **历史只有 5 条**：页面只展示最近 5 条，数据库仍保存完整记录。
4. **Agent 未直接写入**：这是安全设计，必须点击确认。
5. **未来日期被拒绝**：完成记录只能是今天或过去。
6. **数据库正忙**：关闭占用数据库的外部工具后重试。

## 9. 面试可能追问

**问：为什么同时维护最近日期和历史表？** 答：最近日期便于状态查询，历史表用于追溯；事务保证两者一致。

**问：如何证明不会写一半？** 答：测试创建触发器让第二步更新失败，然后确认活动表和物品最近日期都没有变化。

**问：如何防止重复提交？** 答：数据库唯一约束加 `INSERT OR IGNORE`，UI 也会禁用当天已完成的勾选框。

**问：Agent 为什么不能直接写？** 答：写操作会改变用户事实，因此模型只提出动作，用户确认后应用才授权。

## 10. 必须掌握的最少知识

一次完成记录会同时影响活动表和物品表；两步被事务包住；重复提交不会新增数据；Agent 写入前必须确认；历史按日期倒序展示。

## 11. 可自测小题

1. 为什么活动记录和最近日期要在同一事务中？
2. 同一天重复清洗会新增几条记录？
3. `source` 字段解决什么问题？
4. 页面为什么只展示最近 5 条？
5. Agent 什么时候才真正写数据库？

<details><summary>参考答案</summary>

1. 防止只写成功一半。2. 仍然只有一条。3. 标记记录来源。4. 保持卡片简洁，完整数据仍保存。5. 用户点击确认之后。

</details>

## 12. 动手小练习

1. 为一件尚未完成的物品勾选今天已晾晒，再展开历史。
2. 对比手动记录和演示数据的来源文案。
3. 运行 `python -m pytest tests/test_activity_records.py`，观察事务测试通过。

## 13. 本节点术语表

| 术语 | 简单解释 |
|---|---|
| transaction | 一起成功或一起失败的一组操作 |
| commit | 正式保存事务修改 |
| rollback | 撤销事务内尚未完成的修改 |
| idempotent | 重复执行不会重复产生效果 |
| unique constraint | 数据库层面的不重复规则 |
| activity record | 一次具体完成动作的历史记录 |

## 14. 下一节点连接

下一节点会集中处理可靠性、日志、端到端主流程、完整 README 和演示说明，把现有功能整理成可以稳定演示的 MVP。